## 1. Hugging Face Login

#### This imports the token from Kaggle Secrets, so make sure the secret is set correctly before running the code.

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

## 2. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

## 3. Configuration

#### this is sample configuration, you are free to modify any parameter to train the best performing model

In [ ]:
import os
import random
import json
import math
import zipfile
import torch

# Dataset Config
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILE = "Phase_1_Pretraining/Train/train_supreme_court.json"
TEST_FILE  = "Phase_1_Pretraining/Test/test.json"

# ==========================================
# Architecture 4 Prototype (~92M params)
# Parallel Attention-GeGLU with Sandwich-LN
# ==========================================
D_MODEL = 640
N_LAYERS = 16
N_HEADS = 8
D_FF = 2560
MAX_SEQ_LEN = 512       # 512 ensures instant training & 0 OOM errors
TRAIN_BLOCK_SIZE = 512
DROPOUT = 0.1

# Training hyperparameters (Optimized for speed & stability)
TRAIN_TOKENS = 25_000_000   # ~25M tokens trains fast (~20 mins)
VAL_TOKENS   = 1_000_000
BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 4             # Effective batch size = 32
LEARNING_RATE = 4e-4
WEIGHT_DECAY = 0.01
EPOCHS = 1
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

# Generation hyperparameters
TARGET_GENERATION_WORDS = 128
TEMPERATURE = 0.7
TOP_P = 0.90
REPETITION_PENALTY = 1.15

# Output paths
OUTPUT_DIR = "/kaggle/working/cs781_phase1_arch4"
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "alibi_deep_decoder")
PREDICTION_PATH = "/kaggle/working/prediction.json"
SUBMISSION_ZIP = "/kaggle/working/submission.zip"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## 4. Load Tokenizer

In [ ]:
import os
import json
from collections import Counter

SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]
END_OF_WORD = "</w>"

class ByteBPETokenizer:
    def __init__(self, vocab_size=16000):
        self.vocab_size = vocab_size
        self.merges = []
        self.merge_rank = {}
        self.itos = []
        self.stoi = {}

    @staticmethod
    def _word_to_symbols(word):
        byte_str = word.encode("utf-8").decode("latin-1")
        return list(byte_str) + [END_OF_WORD]

    def train(self, corpus_texts):
        word_freq = Counter()
        for text in corpus_texts:
            for word in text.split():
                word_freq[word] += 1

        word_symbols = {w: self._word_to_symbols(w) for w in word_freq}
        vocab = set(SPECIALS)
        vocab.add(END_OF_WORD)
        for i in range(256):
            vocab.add(chr(i))

        target_new_tokens = self.vocab_size - len(vocab)
        merges_done = 0

        while merges_done < target_new_tokens:
            pair_counts = Counter()
            for word, freq in word_freq.items():
                symbols = word_symbols[word]
                for i in range(len(symbols) - 1):
                    pair_counts[(symbols[i], symbols[i + 1])] += freq

            if not pair_counts:
                break
            max_count = max(pair_counts.values())
            if max_count <= 1:
                break

            best_pair = min(p for p, c in pair_counts.items() if c == max_count)
            merged_symbol = best_pair[0] + best_pair[1]
            self.merges.append(best_pair)
            self.merge_rank[best_pair] = merges_done
            vocab.add(merged_symbol)
            merges_done += 1

            for word in word_freq:
                symbols = word_symbols[word]
                new_symbols, i = [], 0
                while i < len(symbols):
                    if (i < len(symbols) - 1
                            and symbols[i] == best_pair[0]
                            and symbols[i + 1] == best_pair[1]):
                        new_symbols.append(merged_symbol)
                        i += 2
                    else:
                        new_symbols.append(symbols[i])
                        i += 1
                word_symbols[word] = new_symbols

        self.itos = (list(SPECIALS) + [END_OF_WORD]
                     + [chr(i) for i in range(256)]
                     + [p[0] + p[1] for p in self.merges])
        self.stoi = {tok: i for i, tok in enumerate(self.itos)}
        self.vocab_size = len(self.itos)
        return self

    def _apply_merges_to_word(self, word):
        symbols = self._word_to_symbols(word)
        while len(symbols) > 1:
            pairs_here = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
            candidates = [(self.merge_rank[p], idx) for idx, p in enumerate(pairs_here) if p in self.merge_rank]
            if not candidates:
                break
            _, idx = min(candidates)
            merged = symbols[idx] + symbols[idx + 1]
            symbols = symbols[:idx] + [merged] + symbols[idx + 2:]
        return symbols

    def encode(self, text):
        ids = []
        for word in text.split():
            for sym in self._apply_merges_to_word(word):
                ids.append(self.stoi.get(sym, self.stoi["<unk>"]))
        return ids

    def decode(self, ids):
        symbols = [self.itos[i] if i < len(self.itos) else "" for i in ids]
        joined = "".join(symbols)
        words = [w for w in joined.split(END_OF_WORD) if w]
        decoded = [w.encode("latin-1").decode("utf-8", errors="replace") for w in words]
        return " ".join(decoded)

    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"merges": self.merges, "itos": self.itos, "vocab_size": self.vocab_size}, f, ensure_ascii=False)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls(vocab_size=data["vocab_size"])
        tok.itos = data["itos"]
        tok.stoi = {t: i for i, t in enumerate(tok.itos)}
        tok.merges = [tuple(p) for p in data["merges"]]
        tok.merge_rank = {p: i for i, p in enumerate(tok.merges)}
        return tok

## 5. Download Training Data From Hugging Face

In [ ]:
from huggingface_hub import hf_hub_download

train_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TRAIN_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Training file:")
print(train_file_path)

In [ ]:
# Train on legal corpus sample or load existing
TOKENIZER_PATH = "/kaggle/input/datasets/brindaaltaccount/legal-bpe-tokenizer/custom_bpe.json"
if os.path.exists(TOKENIZER_PATH):
    print("Loading saved ByteBPETokenizer...")
    tokenizer = ByteBPETokenizer.load(TOKENIZER_PATH)
else:
    print("Training ByteBPETokenizer on legal corpus sample...")
    with open(train_file_path, "r", encoding="utf-8") as f:
        recs = json.load(f)
    sample_txts = [item["text"] for item in recs[:500] if item.get("text")]
    tokenizer = ByteBPETokenizer(vocab_size=8000)
    tokenizer.train(sample_txts)
    tokenizer.save(TOKENIZER_PATH)
    del recs, sample_txts

print(f"Custom ByteBPETokenizer ready! Vocab size: {tokenizer.vocab_size}")

## 6. Create Train and Validation Tokens Set

In [ ]:
with open(train_file_path,"r", encoding="utf-8") as f:
    records = json.load(f)

train_tokens = []
val_tokens = []


random.seed(SEED)

for data in records:
    text = data["text"]
    tokens = tokenizer.encode(text)

    if random.random() < 0.05:
        if len(val_tokens) < VAL_TOKENS:
            remaining = (VAL_TOKENS - len(val_tokens))
            val_tokens.extend(tokens[:remaining])
    else:
        if len(train_tokens) < TRAIN_TOKENS:
            remaining = (TRAIN_TOKENS - len(train_tokens))
            train_tokens.extend(tokens[:remaining])

    if (len(train_tokens) >= TRAIN_TOKENS and len(val_tokens) >= VAL_TOKENS):
        break

print(f"Train tokens: {len(train_tokens):,}")
print(f"Validation tokens: {len(val_tokens):,}")

## 7. Create Token Blocks

In [ ]:
def create_blocks(tokens, block_size):
    total_length = (len(tokens) // block_size) * block_size
    tokens = tokens[:total_length]
    return [tokens[i:i + block_size] for i in range(0, total_length, block_size)]

train_blocks = create_blocks(train_tokens, TRAIN_BLOCK_SIZE)
val_blocks = create_blocks(val_tokens, TRAIN_BLOCK_SIZE)

del train_tokens, val_tokens, records #to optimise ram

print("Training sequences:", len(train_blocks))
print("Validation sequences:", len(val_blocks))

## 8. Create Datasets

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_dict({"input_ids": train_blocks})
val_dataset = Dataset.from_dict({"input_ids": val_blocks})

print(train_dataset)
print(val_dataset)

## 9. Define The Model

In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class RelativePositionBias(nn.Module):
    """T5-style learned bucketed relative position bias."""
    def __init__(self, num_heads: int, num_buckets: int = 32, max_distance: int = 128):
        super().__init__()
        self.num_heads = num_heads
        self.num_buckets = num_buckets
        self.max_distance = max_distance
        self.relative_attention_bias = nn.Embedding(num_buckets, num_heads)

    def _relative_position_bucket(self, relative_position):
        # Causal relative positions (non-positive)
        rel_pos = -torch.clamp(relative_position, max=0)
        num_buckets = self.num_buckets
        max_dist = self.max_distance
        
        # Half buckets for linear close range, half logarithmic for far range
        max_exact = num_buckets // 2
        is_small = rel_pos < max_exact
        
        scale = (num_buckets - max_exact) / math.log(max_dist / max_exact)
        val_if_large = max_exact + (torch.log(rel_pos.float() / max_exact + 1e-6) * scale).long()
        val_if_large = torch.clamp(val_if_large, max=num_buckets - 1)
        
        return torch.where(is_small, rel_pos, val_if_large)

    def forward(self, qlen: int, klen: int, device):
        q_pos = torch.arange(qlen, dtype=torch.long, device=device)[:, None]
        k_pos = torch.arange(klen, dtype=torch.long, device=device)[None, :]
        rel_pos = k_pos - q_pos
        rp_bucket = self._relative_position_bucket(rel_pos)
        values = self.relative_attention_bias(rp_bucket)  # (qlen, klen, num_heads)
        return values.permute(2, 0, 1).unsqueeze(0)        # (1, num_heads, qlen, klen)


class ParallelCustomBlock(nn.Module):
    """
    CUSTOM INNOVATION:
    1. Parallel execution of Multi-Head Attention and GeGLU Feed-Forward.
    2. Sandwich Normalization: Pre-Norm + Post-Residual Norm.
    """
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        
        # Pre-Norms
        self.norm_attn = nn.LayerNorm(d_model)
        self.norm_ffn = nn.LayerNorm(d_model)
        
        # Sandwich Post-Norm
        self.sandwich_norm = nn.LayerNorm(d_model)
        
        # Attention Projections
        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        # GeGLU Projections
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)
        self.w_up = nn.Linear(d_model, d_ff, bias=False)
        self.w_down = nn.Linear(d_ff, d_model, bias=False)
        
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, rel_bias: torch.Tensor) -> torch.Tensor:
        B, S, D = x.shape
        
        # --- BRANCH 1: Attention ---
        norm_x_attn = self.norm_attn(x)
        qkv = self.qkv_proj(norm_x_attn).chunk(3, dim=-1)
        q, k, v = [t.view(B, S, self.n_heads, self.head_dim).transpose(1, 2) for t in qkv]
        
        # Causal mask with learned relative bias
        causal_mask = torch.triu(torch.full((S, S), float('-inf'), device=x.device), diagonal=1)
        attn_bias = rel_bias[:, :, :S, :S] + causal_mask.unsqueeze(0).unsqueeze(0)
        
        attn_out = F.scaled_dot_product_attention(
            q, k, v, attn_mask=attn_bias, dropout_p=self.dropout.p if self.training else 0.0
        )
        attn_out = self.out_proj(attn_out.transpose(1, 2).contiguous().view(B, S, D))
        
        # --- BRANCH 2: GeGLU FFN (runs in parallel!) ---
        norm_x_ffn = self.norm_ffn(x)
        geglu_gate = F.gelu(self.w_gate(norm_x_ffn))
        geglu_up = self.w_up(norm_x_ffn)
        ffn_out = self.w_down(geglu_gate * geglu_up)
        
        # --- COMBINE: Parallel Residual Sum + Sandwich Norm ---
        combined = x + self.dropout(attn_out) + self.dropout(ffn_out)
        return self.sandwich_norm(combined)


class CustomHybridTransformerLM(nn.Module):
    """
    Architecture 4: Parallel Attention-GeGLU with Sandwich-LN (~98M Parameters)
    Completely distinct from standard GPT/LLaMA/Mistral/BLOOM architectures.
    """
    def __init__(self, vocab_size: int, d_model: int = 768, n_layers: int = 12, 
                 n_heads: int = 12, d_ff: int = 2560, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        
        # Single shared relative position bias table across layers
        self.rel_pos_bias = RelativePositionBias(num_heads=n_heads, num_buckets=32, max_distance=256)
        
        self.blocks = nn.ModuleList([
            ParallelCustomBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight  # Weight tying
        
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, S = idx.shape
        x = self.drop(self.tok_emb(idx))
        
        # Compute relative position bias once for all blocks
        rel_bias = self.rel_pos_bias(S, S, idx.device)
        
        for block in self.blocks:
            x = block(x, rel_bias)
            
        x = self.final_norm(x)
        return self.lm_head(x)

# Instantiate Architecture 4 (Custom Hybrid Parallel Model)
model = CustomHybridTransformerLM(
    vocab_size=tokenizer.vocab_size,
    d_model=D_MODEL,
    n_layers=N_LAYERS,
    n_heads=N_HEADS,
    d_ff=D_FF,
    dropout=DROPOUT
).to(device)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Architecture 4 initialized! Total Parameters: {total_params:.2f}M (Target: 50M-300M)")

## 10. Training Arguments and Training

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import time

class PretrainDataset(Dataset):
    def __init__(self, blocks):
        self.blocks = [torch.tensor(b, dtype=torch.long) for b in blocks]
    def __len__(self):
        return len(self.blocks)
    def __getitem__(self, idx):
        return self.blocks[idx]

train_ds = PretrainDataset(train_blocks)
val_ds   = PretrainDataset(val_blocks)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

total_steps = (len(train_loader) // GRAD_ACCUM_STEPS) * EPOCHS
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.05, betas=(0.9, 0.95))
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=3e-5)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

print(f"Starting training on {device} over {total_steps} optimizer steps...")
model.train()
step_count = 0
running_loss = 0.0
train_steps, train_loss = [], []
start_time = time.time()

for epoch in range(EPOCHS):
    for idx, batch_ids in enumerate(train_loader):
        batch_ids = batch_ids.to(device)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(batch_ids)
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = batch_ids[..., 1:].contiguous()
            loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1)) / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        running_loss += loss.item() * GRAD_ACCUM_STEPS

        if (idx + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            step_count += 1

            if step_count % 100 == 0:
                avg_l = running_loss / 100
                train_steps.append(step_count)
                train_loss.append(avg_l)
                print(f"Step {step_count:5d}/{total_steps} | Loss: {avg_l:.4f} | Elapsed: {(time.time()-start_time)/60:.1f}m")
                running_loss = 0.0

print(f"Training completed in {(time.time()-start_time)/3600:.2f} hours.")

## 11. Perplexity

In [ ]:
import os

# Create the target directory so torch can save into it
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
cfg = {
    "d_model": getattr(globals(), "D_MODEL", 768),
    "n_layers": getattr(globals(), "N_LAYERS", 12),
    "n_heads": getattr(globals(), "N_HEADS", 12),
    "d_ff": getattr(globals(), "D_FF", 2560),
    "dropout": getattr(globals(), "DROPOUT", 0.1),
    "vocab_size": tokenizer.vocab_size
}
# Save the trained model weights and tokenizer for Phases 2–7
torch.save({
    "model_state_dict": model.state_dict(),
    "config": cfg,
    "vocab_size": tokenizer.vocab_size
}, os.path.join(MODEL_OUTPUT_DIR, "decoder_lm.pt"))

tokenizer.save(os.path.join(MODEL_OUTPUT_DIR, "custom_bpe.json"))

print(f"SUCCESS! Model checkpoint and tokenizer saved to: {MODEL_OUTPUT_DIR}")

In [ ]:
model.eval()
val_loss_sum = 0.0
val_count = 0

print("Evaluating validation perplexity...")
with torch.no_grad():
    for v_batch in val_loader:
        v_batch = v_batch.to(device)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            v_logits = model(v_batch)
            v_loss = F.cross_entropy(
                v_logits[..., :-1, :].contiguous().view(-1, tokenizer.vocab_size),
                v_batch[..., 1:].contiguous().view(-1)
            )
        val_loss_sum += v_loss.item()
        val_count += 1

eval_loss = val_loss_sum / max(1, val_count)
perplexity = math.exp(eval_loss)
print(f"Validation Loss: {eval_loss:.4f}")
print(f"Validation Perplexity: {perplexity:.2f}")

# Save model and tokenizer for Phases 2–7


## 13. Plot Training and Validation Loss

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

# Plot training loss curve
if train_steps:
    plt.plot(train_steps, train_loss, label="Training Loss", color="#1f77b4", linewidth=2)

# Plot validation loss evaluated at the end
if 'eval_loss' in locals() and train_steps:
    final_step = train_steps[-1]
    plt.scatter([final_step], [eval_loss], color="red", s=100, zorder=5, 
                label=f"Final Val Loss ({eval_loss:.2f})")
    plt.axhline(y=eval_loss, color="red", linestyle="--", alpha=0.5, 
                label=f"Val Loss Baseline ({eval_loss:.2f})")

plt.xlabel("Optimizer Steps", fontsize=12)
plt.ylabel("Cross-Entropy Loss", fontsize=12)
plt.title("Phase 1: Training & Validation Loss (Assignment 3 DecoderLM)", fontsize=13)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig("/kaggle/working/cs781_phase1_arch4/alibi_deep_decoder", dpi=200)
plt.show()


## 14. Save Final Model to Kaggle Output

## 15. Push Model to Hugging Face Hub

## 16. Load Test Data from HuggingFace

In [ ]:
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Test file:", test_file_path)

## 17. Load Test Prompts

In [ ]:
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Number of test examples:", len(test_data))

## 18. Generate 128 Words
#### (words, not tokens)

In [ ]:
def generate_128_words(prompt_text, model, tokenizer, device, target_words=128):
    """
    High-speed 128-word autoregressive generation.
    Stops as soon as 128 words are produced.
    """
    model.eval()
    
    # 1. Encode prompt and keep only the last 256 tokens for fast attention computation
    input_ids = tokenizer.encode(prompt_text)
    if len(input_ids) > 256:
        input_ids = input_ids[-256:]
        
    generated_ids = []
    curr_ids = list(input_ids)
    
    # Pre-allocate input tensor
    x = torch.tensor([curr_ids], dtype=torch.long, device=device)
    
    with torch.no_grad():
        # 150 tokens is plenty to reach 128 words
        for step in range(160):
            # Keep context window under 384 tokens for lightning-fast forward pass
            context = x if x.size(1) <= 384 else x[:, -384:]
            
            logits = model(context)
            next_logits = logits[0, -1, :] / 0.7  # Temperature = 0.7
            
            # Fast repetition penalty on recent 20 generated tokens
            if generated_ids:
                recent_tokens = set(generated_ids[-20:])
                for tok in recent_tokens:
                    next_logits[tok] /= 1.15
            
            # Top-p (nucleus) sampling (top 90%)
            sorted_logits, sorted_indices = torch.sort(next_logits, descending=True)
            cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
            sorted_indices_to_remove = cumulative_probs > 0.90
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0
            indices_to_remove = sorted_indices[sorted_indices_to_remove]
            next_logits[indices_to_remove] = -float('Inf')
            
            probs = torch.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()
            
            generated_ids.append(next_token)
            
            # Append token to input tensor
            next_tensor = torch.tensor([[next_token]], dtype=torch.long, device=device)
            x = torch.cat([x, next_tensor], dim=1)
            
            # Check word count every 10 tokens to break early
            if step >= 110 and step % 10 == 0:
                gen_text = tokenizer.decode(generated_ids)
                if len(gen_text.strip().split()) >= target_words:
                    break
                    
    # 2. Decode and guarantee strictly 128 words
    full_gen_text = tokenizer.decode(generated_ids).strip()
    words = full_gen_text.split()
    
    if len(words) >= target_words:
        return " ".join(words[:target_words])
    
    # In the rare case of fewer words, repeat until strictly target_words
    if not words:
        words = ["the", "court", "held", "that"]
    while len(words) < target_words:
        words.extend(words[:target_words - len(words)])
        
    return " ".join(words[:target_words])

## 20. Generate Predictions for the Test Set

In [ ]:
import os
import json

# Auto-resume: load existing predictions so you NEVER lose progress if session restarts
predictions = {}
if os.path.exists(PREDICTION_PATH):
    try:
        with open(PREDICTION_PATH, "r", encoding="utf-8") as f:
            predictions = json.load(f)
        print(f"Resuming! Found {len(predictions)} already completed predictions.")
    except Exception:
        predictions = {}

items = list(test_data.items())
total = len(items)

# Filter out already generated samples
remaining_items = [(ex_id, rec) for ex_id, rec in items if str(ex_id) not in predictions]
print(f"Total: {total} | Already done: {len(predictions)} | Remaining: {len(remaining_items)}")

model.eval()

for idx, (ex_id, rec) in enumerate(remaining_items):
    # Generates with target words
    continuation = generate_128_words(rec["text"], model, tokenizer, device, target_words=TARGET_GENERATION_WORDS)
    predictions[str(ex_id)] = {"text": continuation}
    
    # Auto-save to disk every 25 samples so you never lose progress!
    if (idx + 1) % 25 == 0 or (idx + 1) == len(remaining_items):
        with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
            json.dump(predictions, f, indent=2, ensure_ascii=False)
        print(f"Saved progress: {len(predictions)}/{total} ({len(predictions)/total*100:.1f}%)")

print("Inference completed! All samples generated and saved.")

## 21. Save prediction.json

In [ ]:
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

if os.path.exists(SUBMISSION_ZIP):
    os.remove(SUBMISSION_ZIP)
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

## 22. Create CodaBench Submission ZIP

In [ ]:
with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zipf:
    contents = zipf.namelist()
    print("Archive contents:", contents)
    assert contents == ["prediction.json"], "ERROR: Archive must contain ONLY prediction.json at root!"
print("\nSUCCESS! /kaggle/working/submission.zip is ready for Codabench submission.")